In [18]:
import sys, importlib
sys.path.insert(0, "../src")
from pyspark.sql import functions as F
import main, m1
importlib.reload(m1)

<module 'm1' from 'c:\\Users\\User\\Documents\\GitHub\\projeto_compar_77655_73107\\compar-spark-docker\\notebooks\\../src\\m1.py'>

In [19]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .master("local[*]")
    .appName("dev-m1")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())

In [20]:
trips = main.load_trips(spark, "../databases", [2025], 1.0)
flagged = m1.flag_invalid(trips)
trips_clean = m1.clean(flagged)

In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")
import importlib, o1
importlib.reload(o1)

zones = main.load_zones(spark, "../databases/taxi_zone_lookup.csv")
base = o1.addFarePerKm(trips_clean)
stats1 = o1.bucketStats(base)
flagged1 = o1.flagAnomalies(base, stats1)
rates = o1.anomaly_rates(flagged1, zones)
rates.show(50, truncate=False)

+--------------+------------+----------+-------+---------+---------------------+
|pickup_borough|payment_type|is_weekend|trips  |anomalies|anomaly_rate         |
+--------------+------------+----------+-------+---------+---------------------+
|Bronx         |1           |false     |4349   |62       |0.014256150839273396 |
|Bronx         |1           |true      |1023   |16       |0.015640273704789834 |
|Bronx         |2           |false     |329    |11       |0.03343465045592705  |
|Bronx         |2           |true      |103    |1        |0.009708737864077669 |
|Bronx         |3           |false     |19     |0        |0.0                  |
|Bronx         |3           |true      |8      |1        |0.125                |
|Bronx         |4           |false     |36     |0        |0.0                  |
|Bronx         |4           |true      |9      |0        |0.0                  |
|Brooklyn      |1           |false     |17279  |345      |0.019966433242664505 |
|Brooklyn      |1           

In [25]:
print(flagged1.count(), trips_clean.count())

flagged1.agg(F.avg(F.col("is_anomaly").cast("int")).alias("rate")).show()

stats1.orderBy(F.desc("std_fpk")).show(5)

rates.filter(F.col("pickup_borough").isNull()).show()

stats = m1.cleaning_stats(flagged)
stats.show()

2791183 2791183
+--------------------+
|                rate|
+--------------------+
|0.002516853964788407|
+--------------------+

+------------+----+------------------+-----------------+
|PULocationID|hour|          mean_fpk|          std_fpk|
+------------+----+------------------+-----------------+
|         223|  23|2545.4668062127366| 8425.83389695662|
|          22|  20| 4629.246556845801|6541.320139068359|
|         138|  12| 75.35224800849409|4738.976280932694|
|         195|  20|1795.4219693350126|2929.452682326221|
|           1|  15|  2023.80121397173|2852.806092886677|
+------------+----+------------------+-----------------+
only showing top 5 rows
+--------------+------------+----------+-----+---------+------------+
|pickup_borough|payment_type|is_weekend|trips|anomalies|anomaly_rate|
+--------------+------------+----------+-----+---------+------------+
+--------------+------------+----------+-----+---------+------------+

+----+-------+------------+------------+----------